# Experiment - Random Forests
Name: Shantanu Shaji

PRN: 24070126165

---
Title: Implementation of RandomForestsClassifier

Aim: To apply the Ensemble technique to modern datasets for predictions

---

**Objective** Students will learn:

- The basic concept of ensemble learning and how it improves classification performance.

- The working principle of the Random Forest algorithm based on bagging and decision trees.

- How multiple decision trees are combined to form a strong classifier.

- Implementing the Random Forest algorithm for classification tasks.

- Evaluating the model's performance using relevant metrics such as accuracy, precision, recall, and F1-score.

- Comparing the performance of Random Forest with a single Decision Tree classifier.

---

**Problem Statement**

Use the given dataset(s) to demonstrate the application of the Random Forest algorithm for classification tasks. The task is to improve classification performance by combining multiple decision trees into a strong classifier using the Random Forest technique, and compare its performance with an individual classifier.

# Stroke Analysis Dataset



## Load dataset and EDA

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google.colab import userdata
path = userdata.get('sml_ds')

In [4]:
df = pd.read_csv(f'{path}/rf_ds.csv')

In [5]:
df.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
2,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
3,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
4,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1


### Analysis - basic

In [6]:
df.shape

(4981, 11)

In [7]:
df.describe()

,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,4981.000000,4981.000000,4981.000000,4981.000000,4981.000000,4981.000000
mean,43.419859,0.096165,0.055210,105.943562,28.498173,0.049789
std,22.662755,0.294848,0.228412,45.075373,6.790464,0.217531
min,0.080000,0.000000,0.000000,55.120000,14.000000,0.000000
25%,25.000000,0.000000,0.000000,77.230000,23.700000,0.000000
50%,45.000000,0.000000,0.000000,91.850000,28.100000,0.000000
75%,61.000000,0.000000,0.000000,113.860000,32.600000,0.000000
max,82.000000,1.000000,1.000000,271.740000,48.900000,1.000000


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4981 entries, 0 to 4980
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             4981 non-null   object 
 1   age                4981 non-null   float64
 2   hypertension       4981 non-null   int64  
 3   heart_disease      4981 non-null   int64  
 4   ever_married       4981 non-null   object 
 5   work_type          4981 non-null   object 
 6   Residence_type     4981 non-null   object 
 7   avg_glucose_level  4981 non-null   float64
 8   bmi                4981 non-null   float64
 9   smoking_status     4981 non-null   object 
 10  stroke             4981 non-null   int64  
dtypes: float64(3), int64(3), object(5)
memory usage: 428.2+ KB


In [9]:
df["stroke"].value_counts() #Classes are skewed. Thus Rnadom Forest will be good.

,count
stroke,
0,4733
1,248


In [10]:
print(df.isnull().sum()) #no null values

gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64


### Categorical data analysis

In [11]:
cat_df = df.select_dtypes(include=['object', 'category'])
cat_df.columns

Index(['gender', 'ever_married', 'work_type', 'Residence_type',
       'smoking_status'],
      dtype='object')

In [12]:
#Frequency distribution of work_type
df['work_type'].value_counts()

,count
work_type,
Private,2860
Self-employed,804
children,673
Govt_job,644


In [13]:
df['smoking_status'].value_counts() #ordinal

,count
smoking_status,
never smoked,1838
Unknown,1500
formerly smoked,867
smokes,776


In [14]:
# categorical data encoding - label encoding
df = pd.get_dummies(df, drop_first=True)
df.shape

(4981, 15)

## df and splitting

In [15]:
df.head()

,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,gender_Male,ever_married_Yes,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,228.69,36.6,1,True,True,True,False,False,True,True,False,False
1,80.0,0,1,105.92,32.5,1,True,True,True,False,False,False,False,True,False
2,49.0,0,0,171.23,34.4,1,False,True,True,False,False,True,False,False,True
3,79.0,1,0,174.12,24.0,1,False,True,False,True,False,False,False,True,False
4,81.0,0,0,186.21,29.0,1,True,True,True,False,False,True,True,False,False


In [16]:
x = df.drop("stroke", axis=1)
y = df["stroke"]

### Train-test splits

In [17]:
from sklearn.model_selection import train_test_split

In [18]:
# 60-40 split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.4, random_state=42)

#70-30 split
x_train1, x_test1, y_train1, y_test1 = train_test_split(x,y,test_size = 0.3, random_state = 42)

### Implementing the model

In [19]:
from sklearn.ensemble import RandomForestClassifier

In [20]:
# model 1 --> n_estimators = 100 , n_jobs = None ) -- default
model1 = RandomForestClassifier(n_estimators=100, n_jobs=None, class_weight ="balanced")
model1.fit(x_train, y_train)

RandomForestClassifier(class_weight='balanced')

In [21]:
from sklearn.metrics import classification_report

y_pred1 = model1.predict(x_test)

print("Classification Report for model1 (60-40 split):\n", classification_report(y_test, y_pred1))

Classification Report for model1 (60-40 split):
               precision    recall  f1-score   support

           0       0.95      1.00      0.97      1890
           1       0.00      0.00      0.00       103

    accuracy                           0.95      1993
   macro avg       0.47      0.50      0.49      1993
weighted avg       0.90      0.95      0.92      1993



In [22]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint


param_dist = {
    'n_estimators': randint(50, 200),
    'n_jobs': [-1, None],
}

rf_model = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    scoring='accuracy',
    random_state=1,

    n_jobs=-1
)

random_search.fit(x_train, y_train)
print("RandomizedSearchCV fitting complete.")

# Print the best parameters and best score
print("Best parameters found: ", random_search.best_params_)
print("Best accuracy found: ", random_search.best_score_)

best = random_search.best_estimator_

RandomizedSearchCV fitting complete.
Best parameters found:  {'n_estimators': 194, 'n_jobs': None}
Best accuracy found:  0.9514725242713009


In [23]:
random_search.get_params

<bound method BaseEstimator.get_params of RandomizedSearchCV(estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
                   param_distributions={'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7a5b47d07680>,
                                        'n_jobs': [-1, None]},
                   random_state=1, scoring='accuracy')>

In [24]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [194, 192, 190, 185, 200, 198],
    'n_jobs': [-1, None],
    'class_weight': ['balanced']
}

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

# Fit GridSearchCV to the training data
print("Fitting GridSearchCV...")
grid_search.fit(x_train, y_train)
print("GridSearchCV fitting complete.")

# Print the best parameters and best score found by GridSearchCV
print("Best parameters found by GridSearchCV: ", grid_search.best_params_)
print("Best accuracy found by GridSearchCV: ", grid_search.best_score_)

# Store the best model
best_rf_model_grid = grid_search.best_estimator_

Fitting GridSearchCV...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
GridSearchCV fitting complete.
Best parameters found by GridSearchCV:  {'class_weight': 'balanced', 'n_estimators': 194, 'n_jobs': -1}
Best accuracy found by GridSearchCV:  0.9514725568942438


In [25]:
y_pred_grid = best_rf_model_grid.predict(x_test)
print("Classification Report\n",
      classification_report(y_test, y_pred_grid))

Classification Report
               precision    recall  f1-score   support

           0       0.95      1.00      0.97      1890
           1       0.00      0.00      0.00       103

    accuracy                           0.95      1993
   macro avg       0.47      0.50      0.49      1993
weighted avg       0.90      0.95      0.92      1993

